<a href="https://colab.research.google.com/github/todd-jang/AIFFEL_quest_rs/blob/main/KITTI_RetinaNet_Autonomous_Driving_chatGPT-Submission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚗 KITTI 기반 RetinaNet 자율주행 Object Detection 프로젝트

**제출 루브릭 대응형 Colab Notebook**

### 목표
1. KITTI 데이터셋 구조와 클래스 분포를 분석한다.
2. KITTI annotation을 RetinaNet 입력 형식으로 가공한다.
3. **RetinaNet-ResNet50-FPN을 실제로 KITTI 데이터로 fine-tuning**한다.
4. Ground Truth와 Detection 결과의 bounding box를 시각화한다.
5. 자율주행 보조 규칙을 적용해 `Go / Stop`을 판정한다.
6. 제공된 `stop_1~5.png`, `go_1~5.png`가 있으면 10장 테스트에서 **90점 이상인지 실제로 측정**한다.

> ⚠️ 90% 이상의 결과를 미리 만들어 넣지 않습니다. 실제 Colab 실행 결과를 그대로 평가합니다.


## 루브릭 대응표

| 평가 기준 | 본 Notebook |
|---|---|
| KITTI 분석 | 클래스 분포, 객체 수, bounding box 통계, 샘플 시각화 |
| 데이터 가공 | KITTI → torchvision RetinaNet target 변환 |
| RetinaNet 학습 | **RetinaNet ResNet50 + FPN + KITTI fine-tuning** |
| Bounding box 시각화 | GT / Prediction overlay |
| 자율주행 테스트 | 사람 또는 근접 차량 → Stop, 그 외 → Go |
| 90% 이상 정확도 | 10개 제공 평가 이미지에 대해 실제 점수 계산 |

### 중요한 수정점

기존 코드에는 RetinaNet 학습 루프가 존재하지만, 프로젝트의 `self_drive_assist()`에서는 **학습한 KITTI RetinaNet이 아니라 COCO 사전학습 RetinaNet을 별도로 불러와 추론**하는 부분이 있습니다.

따라서 이 제출본에서는 **학습한 KITTI RetinaNet → 평가 → 자율주행 보조 시스템**으로 하나의 모델 흐름을 사용합니다.


In [ ]:
# 1. Colab 환경 확인
import os, random, math, time, json, copy
from pathlib import Path

import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

print("PyTorch     :", torch.__version__)
print("Torchvision :", torchvision.__version__)
print("CUDA 사용   :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
    print("⚠️ GPU가 없습니다. Colab에서 런타임 → 런타임 유형 변경 → T4 GPU를 선택하세요.")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ROOT = Path("/content/object_detection/data")
ROOT.mkdir(parents=True, exist_ok=True)
print("DATA ROOT:", ROOT)


## 2. KITTI 데이터셋 다운로드

`torchvision.datasets.Kitti`를 사용합니다.

KITTI 2D Object Detection 데이터는 이미지와 annotation을 함께 사용하며, annotation에는 객체 종류와 bounding box 좌표가 포함됩니다.

> 처음 실행할 때 데이터 다운로드에 시간이 오래 걸릴 수 있습니다.


In [ ]:
# 2. KITTI 다운로드
# 이미 다운로드되어 있으면 download=True여도 다시 받지 않습니다.

kitti_train_raw = torchvision.datasets.Kitti(
    root=str(ROOT), train=True, download=True
)

print("KITTI train samples:", len(kitti_train_raw))
print("첫 샘플 이미지 타입:", type(kitti_train_raw[0][0]))
print("첫 샘플 annotation 개수:", len(kitti_train_raw[0][1]))
print("첫 annotation 예시:", kitti_train_raw[0][1][0] if kitti_train_raw[0][1] else "empty")


## 3. KITTI 데이터 구조 분석

이번 프로젝트에서는 KITTI의 다음 정보를 사용합니다.

- `type`: 객체 클래스
- `bbox`: `(x_min, y_min, x_max, y_max)` pixel 좌표
- `DontCare`: 학습 대상에서 제외

RetinaNet 학습에서는 다음 8개 클래스를 사용합니다.

`Car, Van, Truck, Pedestrian, Person_sitting, Cyclist, Tram, Misc`


In [ ]:
KITTI_CLASSES = [
    "Car", "Van", "Truck", "Pedestrian",
    "Person_sitting", "Cyclist", "Tram", "Misc"
]
CLASS_TO_ID = {name: i for i, name in enumerate(KITTI_CLASSES)}

# KITTI annotation 원본 클래스 빈도
from collections import Counter

class_counter = Counter()
object_count = 0

ANALYSIS_LIMIT = min(1000, len(kitti_train_raw))

for i in range(ANALYSIS_LIMIT):
    _, target = kitti_train_raw[i]
    for obj in target:
        name = obj["type"]
        if name != "DontCare":
            class_counter[name] += 1
            object_count += 1

print("분석 샘플 수:", ANALYSIS_LIMIT)
print("분석 객체 수:", object_count)
print("클래스별 객체 수:")
for name, count in class_counter.most_common():
    print(f"{name:18s}: {count}")


In [ ]:
# 클래스 분포 시각화
names = list(class_counter.keys())
counts = [class_counter[n] for n in names]

plt.figure(figsize=(12, 5))
plt.bar(names, counts)
plt.title("KITTI class distribution")
plt.xlabel("Class")
plt.ylabel("Object count")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


## 4. KITTI Ground Truth Bounding Box 확인

학습 전에 annotation이 실제 이미지의 객체 위치와 일치하는지 확인합니다.

이 셀의 결과는 루브릭의 **“KITTI 데이터셋 구조와 내용을 파악하고 필요한 데이터셋 가공”** 및 **“바운딩박스가 정확히 표시된 시각화 이미지”**를 증명하는 자료로 사용할 수 있습니다.


In [ ]:
def show_kitti_gt(dataset, indices=(0, 1, 2), figsize=(18, 6)):
    fig, axes = plt.subplots(1, len(indices), figsize=figsize)
    if len(indices) == 1:
        axes = [axes]

    for ax, idx in zip(axes, indices):
        image, target = dataset[idx]
        ax.imshow(image)

        for obj in target:
            name = obj["type"]
            if name not in CLASS_TO_ID:
                continue
            x1, y1, x2, y2 = obj["bbox"]
            rect = plt.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                fill=False, linewidth=2
            )
            ax.add_patch(rect)
            ax.text(x1, y1, name, fontsize=9,
                    bbox=dict(alpha=0.6, pad=2))

        ax.set_title(f"KITTI sample {idx}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

show_kitti_gt(kitti_train_raw, (0, 10, 20))


## 5. KITTI → RetinaNet 데이터 가공

torchvision RetinaNet은 이미지와 함께 다음 형태의 target을 받습니다.

```text
target = {
    "boxes": Tensor[N, 4],   # xyxy
    "labels": Tensor[N]
}
```

KITTI의 `type`, `bbox`를 이 형식으로 변환합니다.


In [ ]:
class KITTIRetinaDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, indices=None, train=False):
        self.base = base_dataset
        self.indices = list(range(len(base_dataset))) if indices is None else list(indices)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        image, raw_target = self.base[real_idx]

        image = torchvision.transforms.functional.to_tensor(image)

        boxes, labels = [], []

        for obj in raw_target:
            name = obj["type"]
            if name not in CLASS_TO_ID:
                continue

            x1, y1, x2, y2 = obj["bbox"]

            # 잘못된/빈 박스 제거
            if x2 <= x1 or y2 <= y1:
                continue

            boxes.append([x1, y1, x2, y2])
            labels.append(CLASS_TO_ID[name])

        if boxes:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
        else:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([real_idx])
        }

        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))


In [ ]:
# 6. Train / Validation split
all_indices = list(range(len(kitti_train_raw)))
random.Random(SEED).shuffle(all_indices)

# 최종 제출에서는 False로 바꾸면 전체 KITTI train split을 사용합니다.
FAST_MODE = True

if FAST_MODE:
    TRAIN_LIMIT = min(600, len(all_indices))
    VAL_LIMIT = min(150, len(all_indices) - TRAIN_LIMIT)
else:
    TRAIN_LIMIT = int(len(all_indices) * 0.9)
    VAL_LIMIT = len(all_indices) - TRAIN_LIMIT

train_indices = all_indices[:TRAIN_LIMIT]
val_indices = all_indices[TRAIN_LIMIT:TRAIN_LIMIT + VAL_LIMIT]

train_dataset = KITTIRetinaDataset(kitti_train_raw, train_indices, train=True)
val_dataset = KITTIRetinaDataset(kitti_train_raw, val_indices, train=False)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=2, shuffle=True,
    num_workers=2, collate_fn=collate_fn, pin_memory=torch.cuda.is_available()
)

val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=1, shuffle=False,
    num_workers=2, collate_fn=collate_fn, pin_memory=torch.cuda.is_available()
)

print("Train:", len(train_dataset))
print("Val  :", len(val_dataset))


## 7. RetinaNet 모델 구성

**RetinaNet + ResNet50-FPN**을 사용합니다.

COCO 사전학습 backbone에서 시작해 KITTI 8개 클래스를 학습하도록 RetinaNet classification head를 교체합니다.

즉, 여기부터는 단순히 COCO 모델을 추론하는 것이 아니라 **KITTI 데이터로 실제 fine-tuning**합니다.


In [ ]:
from torchvision.models.detection import (
    retinanet_resnet50_fpn,
    RetinaNet_ResNet50_FPN_Weights
)
from torchvision.models.detection.retinanet import RetinaNetClassificationHead

NUM_CLASSES = len(KITTI_CLASSES)

# COCO pretrained RetinaNet을 backbone 초기값으로 사용
model = retinanet_resnet50_fpn(
    weights=RetinaNet_ResNet50_FPN_Weights.DEFAULT
)

old_head = model.head.classification_head

model.head.classification_head = RetinaNetClassificationHead(
    in_channels=model.backbone.out_channels,
    num_anchors=old_head.num_anchors,
    num_classes=NUM_CLASSES,
    prior_probability=0.01
)

model.to(DEVICE)

print(model.__class__.__name__)
print("KITTI classes:", NUM_CLASSES)
print("Device:", DEVICE)


## 8. RetinaNet 실제 학습

학습 loss는 torchvision RetinaNet 내부의 focal loss 기반 classification loss와 bounding box regression loss를 사용합니다.

### 권장 설정
- Colab T4: `EPOCHS = 5~10`
- 빠른 동작 확인: `FAST_MODE=True`
- 최종 제출: `FAST_MODE=False` + 충분한 epoch

⚠️ 학습 시간은 GPU 종류와 데이터 수에 따라 크게 달라집니다.


In [ ]:
EPOCHS = 3 if FAST_MODE else 8
LR = 0.0001

optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
    momentum=0.9,
    weight_decay=0.0005
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, step_size=max(1, EPOCHS // 2), gamma=0.1
)

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    steps = 0

    for images, targets in loader:
        images = [img.to(device) for img in images]
        targets = [
            {k: v.to(device) if torch.is_tensor(v) else v for k, v in t.items()}
            for t in targets
        ]

        optimizer.zero_grad(set_to_none=True)
        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()

        total_loss += loss.item()
        steps += 1

    return total_loss / max(1, steps)


history = []

for epoch in range(EPOCHS):
    start = time.time()
    train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
    scheduler.step()

    history.append(train_loss)
    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"loss={train_loss:.4f} | "
        f"lr={scheduler.get_last_lr()[0]:.6f} | "
        f"time={time.time()-start:.1f}s"
    )


In [ ]:
# 학습 loss curve
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(history)+1), history, marker="o")
plt.title("RetinaNet training loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# 학습 모델 저장
CHECKPOINT = ROOT / "retinanet_kitti_final.pth"

torch.save({
    "model_state_dict": model.state_dict(),
    "class_to_id": CLASS_TO_ID,
    "classes": KITTI_CLASSES,
    "epoch": EPOCHS,
    "loss_history": history,
}, CHECKPOINT)

print("Saved:", CHECKPOINT)


## 9. 학습한 RetinaNet으로 Detection 시각화

이 셀에서는 반드시 **방금 KITTI로 학습한 `model`**을 사용합니다.

Prediction bounding box와 confidence를 이미지 위에 표시합니다.


In [ ]:
ID_TO_CLASS = {v: k for k, v in CLASS_TO_ID.items()}

@torch.no_grad()
def predict_image(model, image, score_threshold=0.35):
    model.eval()

    tensor = torchvision.transforms.functional.to_tensor(image).to(DEVICE)
    output = model([tensor])[0]

    keep = output["scores"] >= score_threshold

    boxes = output["boxes"][keep].detach().cpu()
    labels = output["labels"][keep].detach().cpu()
    scores = output["scores"][keep].detach().cpu()

    return boxes, labels, scores


def visualize_prediction(model, dataset, dataset_pos=0, score_threshold=0.35):
    image, target = dataset[dataset_pos]

    # 원본 PIL 이미지는 base dataset에서 다시 가져옴
    real_idx = dataset.indices[dataset_pos]
    pil_image, _ = dataset.base[real_idx]

    boxes, labels, scores = predict_image(
        model, pil_image, score_threshold
    )

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(pil_image)

    # Ground Truth
    gt = target["boxes"]
    for box, label in zip(gt, target["labels"]):
        x1, y1, x2, y2 = box.tolist()
        rect = plt.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            fill=False, linewidth=2, linestyle="--"
        )
        ax.add_patch(rect)
        ax.text(
            x1, y1, "GT:" + ID_TO_CLASS[int(label)],
            fontsize=9, bbox=dict(alpha=0.6, pad=2)
        )

    # Prediction
    for box, label, score in zip(boxes, labels, scores):
        x1, y1, x2, y2 = box.tolist()
        rect = plt.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            fill=False, linewidth=2
        )
        ax.add_patch(rect)
        ax.text(
            x1, max(0, y1-12),
            f"Pred:{ID_TO_CLASS[int(label)]} {float(score):.2f}",
            fontsize=9, bbox=dict(alpha=0.6, pad=2)
        )

    ax.set_title("KITTI Ground Truth / RetinaNet Prediction")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    return boxes, labels, scores


visualize_prediction(model, val_dataset, 0)
visualize_prediction(model, val_dataset, min(5, len(val_dataset)-1))


## 10. 간단한 Detection 성능 평가

정식 object detection 평가는 mAP가 가장 적절하지만, 프로젝트 제출에서 모델이 실제로 객체를 얼마나 맞추는지 확인할 수 있도록 IoU 0.5 기준 precision / recall도 계산합니다.

- IoU ≥ 0.5 → TP
- 예측은 있는데 대응 GT가 없으면 FP
- GT가 있는데 대응 prediction이 없으면 FN


In [ ]:
def box_iou_one_to_many(box, boxes):
    if len(boxes) == 0:
        return torch.zeros((0,))
    x1 = torch.maximum(box[0], boxes[:, 0])
    y1 = torch.maximum(box[1], boxes[:, 1])
    x2 = torch.minimum(box[2], boxes[:, 2])
    y2 = torch.minimum(box[3], boxes[:, 3])

    inter = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)
    area1 = (box[2]-box[0]).clamp(min=0) * (box[3]-box[1]).clamp(min=0)
    area2 = (boxes[:,2]-boxes[:,0]).clamp(min=0) * (boxes[:,3]-boxes[:,1]).clamp(min=0)
    union = area1 + area2 - inter + 1e-6
    return inter / union


@torch.no_grad()
def detection_precision_recall(model, dataset, max_samples=100, score_threshold=0.35, iou_threshold=0.5):
    tp = fp = fn = 0
    n = min(len(dataset), max_samples)

    for i in range(n):
        image, target = dataset[i]
        real_idx = dataset.indices[i]
        pil_image, _ = dataset.base[real_idx]

        pred_boxes, pred_labels, _ = predict_image(
            model, pil_image, score_threshold
        )

        gt_boxes = target["boxes"]
        gt_labels = target["labels"]
        matched = set()

        for pb, pl in zip(pred_boxes, pred_labels):
            candidates = [
                j for j, gl in enumerate(gt_labels)
                if int(gl) == int(pl) and j not in matched
            ]

            if not candidates:
                fp += 1
                continue

            ious = box_iou_one_to_many(pb, gt_boxes[candidates])
            best_pos = int(torch.argmax(ious))
            best_iou = float(ious[best_pos])

            if best_iou >= iou_threshold:
                tp += 1
                matched.add(candidates[best_pos])
            else:
                fp += 1

        fn += len(gt_boxes) - len(matched)

    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 2 * precision * recall / max(1e-8, precision + recall)

    return {
        "samples": n,
        "TP": tp, "FP": fp, "FN": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


det_metrics = detection_precision_recall(
    model, val_dataset,
    max_samples=min(100, len(val_dataset)),
    score_threshold=0.35
)

print(json.dumps(det_metrics, indent=2))


## 11. 자율주행 보조 시스템

프로젝트 요구사항:

- 사람이 한 명 이상 감지되면 `Stop`
- 차량의 bounding box `width` 또는 `height`가 300px 이상이면 `Stop`
- 그 외에는 `Go`

### 안전 방향

실제 자율주행에서는 단순한 pixel 크기 하나만으로 판단하면 안 됩니다. 여기서는 교육용 프로젝트의 평가 조건을 그대로 구현합니다.


In [ ]:
# KITTI 클래스 기준
PERSON_CLASSES = {
    CLASS_TO_ID["Pedestrian"],
    CLASS_TO_ID["Person_sitting"]
}

VEHICLE_CLASSES = {
    CLASS_TO_ID["Car"],
    CLASS_TO_ID["Van"],
    CLASS_TO_ID["Truck"],
    CLASS_TO_ID["Tram"]
}


@torch.no_grad()
def self_drive_assist(img_path, size_limit=300, score_threshold=0.5):
    image = Image.open(img_path).convert("RGB")

    boxes, labels, scores = predict_image(
        model, image, score_threshold=score_threshold
    )

    decision = "Go"
    reasons = []

    for box, label, score in zip(boxes, labels, scores):
        label = int(label)
        x1, y1, x2, y2 = box.tolist()
        w, h = x2 - x1, y2 - y1

        if label in PERSON_CLASSES:
            decision = "Stop"
            reasons.append(
                f"person detected ({ID_TO_CLASS[label]}, {float(score):.2f})"
            )

        elif label in VEHICLE_CLASSES and max(w, h) >= size_limit:
            decision = "Stop"
            reasons.append(
                f"large vehicle ({ID_TO_CLASS[label]}, "
                f"{w:.0f}x{h:.0f}px, {float(score):.2f})"
            )

    return decision, reasons, boxes, labels, scores


print("self_drive_assist 준비 완료")


In [ ]:
def visualize_drive_decision(img_path, size_limit=300, score_threshold=0.5):
    image = Image.open(img_path).convert("RGB")
    decision, reasons, boxes, labels, scores = self_drive_assist(
        img_path, size_limit=size_limit,
        score_threshold=score_threshold
    )

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(image)

    for box, label, score in zip(boxes, labels, scores):
        x1, y1, x2, y2 = box.tolist()
        w, h = x2-x1, y2-y1

        rect = plt.Rectangle(
            (x1, y1), w, h,
            fill=False, linewidth=2
        )
        ax.add_patch(rect)

        ax.text(
            x1, max(0, y1-12),
            f"{ID_TO_CLASS[int(label)]} {float(score):.2f} [{w:.0f}x{h:.0f}]",
            fontsize=9, bbox=dict(alpha=0.65, pad=2)
        )

    ax.set_title(f"Driving Assist Decision: {decision}")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    print("Decision:", decision)
    if reasons:
        print("Reasons:")
        for r in reasons:
            print(" -", r)


# 예시: KITTI 검증 이미지로 동작 확인
example_real_idx = val_dataset.indices[0]
example_path = Path(kitti_train_raw._image_dir) / f"{kitti_train_raw.images[example_real_idx]}"
print("예시 이미지는 KITTI 내부 경로 구조에 따라 직접 지정하지 않고 아래 셀의 검증 이미지로 확인합니다.")


## 12. 제공된 GO / STOP 평가 이미지 테스트

AIFFEL 프로젝트에서 제공되는 평가 이미지를 다음 위치에 넣으면 됩니다.

```text
/content/object_detection/data/
├── stop_1.png
├── stop_2.png
├── stop_3.png
├── stop_4.png
├── stop_5.png
├── go_1.png
├── go_2.png
├── go_3.png
├── go_4.png
└── go_5.png
```

파일이 없으면 아래 셀은 오류 대신 **평가 이미지가 없다는 상태를 명확히 표시**합니다.


In [ ]:
TEST_DIR = ROOT

TEST_SET = [
    ("stop_1.png", "Stop"),
    ("stop_2.png", "Stop"),
    ("stop_3.png", "Stop"),
    ("stop_4.png", "Stop"),
    ("stop_5.png", "Stop"),
    ("go_1.png", "Go"),
    ("go_2.png", "Go"),
    ("go_3.png", "Go"),
    ("go_4.png", "Go"),
    ("go_5.png", "Go"),
]

available = [
    (name, answer) for name, answer in TEST_SET
    if (TEST_DIR / name).exists()
]

if not available:
    print("⚠️ GO/STOP 평가 이미지가 없습니다.")
    print("다음 10개 파일을", TEST_DIR, "에 업로드한 뒤 이 셀을 다시 실행하세요.")
else:
    correct = 0
    results = []

    for name, answer in available:
        path = TEST_DIR / name
        pred, reasons, boxes, labels, scores = self_drive_assist(
            path, size_limit=300, score_threshold=0.5
        )

        ok = pred == answer
        correct += int(ok)

        results.append({
            "image": name,
            "answer": answer,
            "prediction": pred,
            "correct": ok,
            "reason": "; ".join(reasons) if reasons else "-"
        })

    score = 100 * correct / len(available)

    print(f"테스트 이미지 수: {len(available)}/10")
    print(f"정답 수         : {correct}")
    print(f"정확도(점수)    : {score:.1f}%")

    for r in results:
        mark = "✓" if r["correct"] else "✗"
        print(f"{mark} {r['image']:12s} GT={r['answer']:4s} Pred={r['prediction']:4s} | {r['reason']}")

    if len(available) == 10:
        if score >= 90:
            print("\n🎉 루브릭 목표인 90점 이상을 달성했습니다.")
        else:
            print("\n⚠️ 90점 미만입니다. threshold / 학습 epoch / 데이터 규모를 조정해 재학습하세요.")
    else:
        print("\n⚠️ 10장 전체가 준비되지 않아 최종 100점 평가로 보지 않습니다.")


## 13. 최종 제출 체크리스트

### 제출 전 반드시 확인

- [ ] GPU가 활성화되어 있다.
- [ ] KITTI 데이터 다운로드가 정상 완료되었다.
- [ ] KITTI 클래스 분포 그래프가 출력되었다.
- [ ] Ground Truth bounding box 이미지가 출력되었다.
- [ ] `RetinaNet-ResNet50-FPN` 학습 loss가 출력되었다.
- [ ] `retinanet_kitti_final.pth`가 생성되었다.
- [ ] **학습한 KITTI RetinaNet**으로 prediction bounding box가 출력되었다.
- [ ] Detection precision / recall / F1 결과를 확인했다.
- [ ] `stop_1~5.png`, `go_1~5.png` 10장을 업로드했다.
- [ ] 최종 `Go/Stop` 점수가 **90점 이상**인지 확인했다.

### 최종 발표에서 강조할 문장

> “KITTI annotation을 RetinaNet target 형식으로 변환하고, ResNet50-FPN backbone 기반 RetinaNet을 KITTI 데이터로 fine-tuning하였다. 이후 동일하게 학습된 모델을 자율주행 보조 판단 모듈에 연결하여 사람 검출 및 차량 크기 기반 Stop/Go 의사결정을 수행하였다.”
